In [1]:
import torch

In [2]:
!pip install bm25s

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 3.4 MB/s eta 0:00:00


In [3]:
import bm25s

In [4]:
!pip install langchain-ollama

/usr/lib/python3.12/pty.py:95: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


In [5]:
!pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 57.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0

In [6]:
!pip install langchain_text_splitters

In [7]:
!pip install rank_bm25

In [8]:
!pip install faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.3/135.3 MB 13.7 MB/s eta 0:00:0000:0100:01


In [9]:
import torch

In [10]:
torch.cuda.is_available()

True

In [11]:
"""
Улучшенный модуль для очистки и конвертации викитекста Неолурка (MediaWiki) в чистый Markdown.

Исправлены ключевые проблемы прототипа:
1. Вложенные скобки в изображениях: [[Файл:...[[ссылка]]...]] больше не оставляет мусорные "]]" в тексте.
2. Вложенные шаблоны: цитаты {{Q|...{{nobr|...}}...|Автор}} больше не обрезаются и корректно форматируют автора.
3. Таблицы: синтаксис MediaWiki-таблиц ({| ... |}) корректно очищается от CSS-стилей, а полезный текст сохраняется.
4. Списки: нумерованные списки MediaWiki (# пункт) больше не ломают разметку заголовков Markdown (# Заголовок 1).
5. Заголовки: устойчивость к пробелам и комментариям на концах строк (== Заголовок == ).
6. Ссылки: корректная обработка внешних ссылок [http://site.com Описание] -> [Описание](http://site.com).
7. HTML: декодирование сущностей (&quot;, &nbsp; и др.), удаление комментариев <!-- ... -->, тегов <gallery> и очистка разметки.
"""

import sqlite3
import re
import html
import sys

# Обеспечиваем корректный вывод UTF-8 в консоли Windows
if sys.platform == "win32":
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")


def remove_wiki_images(text: str) -> str:
    """
    Удаляет теги файлов и картинок MediaWiki ([[Файл:...]], [[Изображение:...]])
    с корректным учётом любой глубины вложенных ссылок в подписях.
    """
    prefixes = ("[[файл:", "[[изображение:", "[[file:", "[[image:")
    result = []
    i = 0
    n = len(text)

    while i < n:
        lower_slice = text[i:i + 15].lower()
        if any(lower_slice.startswith(p) for p in prefixes):
            depth = 0
            j = i
            while j < n:
                if text[j:j + 2] == "[[":
                    depth += 1
                    j += 2
                elif text[j:j + 2] == "]]":
                    depth -= 1
                    j += 2
                    if depth <= 0:
                        break
                else:
                    j += 1
            if j < n and text[j] == "\n":
                j += 1
            i = j
        else:
            result.append(text[i])
            i += 1

    return "".join(result)


def process_single_template(template_body: str) -> str:
    """Обрабатывает один плоский (самый внутренний) шаблон MediaWiki."""
    parts = [p.strip() for p in template_body.split('|')]
    if not parts:
        return ""

    name = parts[0].lower()
    args = parts[1:]

    # Зачёркнутый текст: {{s|текст}} -> ~~текст~~
    if name in ('s', 'strike', 'del', 'зачёркнутый', 'зачеркнуто'):
        for arg in args:
            if '=' not in arg:
                return f"~~{arg}~~"
        return ""

    # Неразрывный текст: {{nobr|текст}} -> текст
    if name in ('nobr', 'nowrap'):
        for arg in args:
            if '=' not in arg:
                return arg
        return ""

    # Ссылки на Википедию / классический Лурк: {{w|Статья|Текст}} -> Текст
    if name in ('w', 'wikipedia', 'википедия', 'lurk', 'lm', 'неолурк', 'neolurk'):
        pos_args = [a for a in args if '=' not in a]
        if len(pos_args) >= 2:
            return pos_args[1]
        elif len(pos_args) == 1:
            return pos_args[0]
        return ""

    # Спойлеры: {{spoiler|текст}} или {{spoiler|Заголовок|текст}}
    if name in ('spoiler', 'спойлер'):
        pos_args = [a for a in args if '=' not in a]
        if pos_args:
            return pos_args[-1]
        return ""

    # Цитаты: {{q|текст|автор}} или {{цитата|текст|автор=...}}
    if name in ('q', 'quote', 'цитата', 'высказывание', 'citat'):
        quote_text = ""
        author = ""
        pos_args = []
        for a in args:
            if a.lower().startswith('автор='):
                author = a.split('=', 1)[1].strip()
            elif a.lower().startswith('источник='):
                pass
            elif a.lower().startswith('pre='):
                continue
            elif '=' in a:
                continue
            else:
                pos_args.append(a)

        if pos_args:
            quote_text = pos_args[0]
            if len(pos_args) > 1 and not author:
                author = pos_args[1]

        if not quote_text:
            return ""

        lines = [line.strip() for line in quote_text.splitlines() if line.strip()]
        md_quote = '\n' + '\n'.join(f'> {line}' for line in lines)
        if author:
            md_quote += f'\n> — *{author}*'
        return md_quote + '\n'

    # Языковые шаблоны: {{lang|en|word}}
    if name == 'lang' and len(args) >= 2:
        return args[1]
    if name.startswith('lang-') and args:
        return args[0]

    # Комментарии / всплывающие подсказки: {{comment|текст|подсказка}}
    if name in ('comment', 'подсказка') and args:
        return args[0]

    # Служебные плашки (стабы, навигационные списки, плашки плагиата) удаляем
    return ""


def unwind_templates(text: str, max_depth: int = 25) -> str:
    """Раскрывает шаблоны от самых внутренних к внешним во избежание обрезки."""
    pattern = re.compile(r'\{\{([^{}]+)\}\}')
    for _ in range(max_depth):
        if not pattern.search(text):
            break
        text = pattern.sub(lambda m: process_single_template(m.group(1)), text)

    # Удаляем остаточные незакрытые шаблоны
    text = re.sub(r'\{\{[^{}]*$', '', text)
    text = re.sub(r'^\s*\}\}', '', text, flags=re.MULTILINE)
    return text


def clean_tables(text: str) -> str:
    """
    Очищает таблицы MediaWiki ({| ... |}), извлекая полезное содержимое
    ячеек и удаляя служебные CSS/HTML атрибуты оформления.
    """
    def table_replacer(match):
        table_raw = match.group(0)
        lines = table_raw.splitlines()
        body_lines = []

        for line in lines[1:-1]:
            line = line.strip()
            if not line:
                continue
            if line.startswith('|-'):
                body_lines.append('')
                continue
            if line.startswith(('|', '!')):
                content = line[1:].strip()
                # Удаляем атрибуты вида style="...", width=50% | Текст ячейки
                if '|' in content and not content.startswith('{'):
                    parts = content.split('|')
                    if any(attr in parts[0].lower() for attr in ('class=', 'style=', 'bgcolor=', 'colspan=', 'rowspan=', 'align=', 'width=')):
                        content = '|'.join(parts[1:]).strip()
                if content:
                    body_lines.append(content)
            else:
                body_lines.append(line)

        return '\n' + '\n'.join(body_lines) + '\n'

    return re.sub(r'\{\|.*?\n\|\}', table_replacer, text, flags=re.DOTALL)


def convert_headers(text: str) -> str:
    """Преобразует заголовки == Header == в Markdown ## Header устойчиво к концевым пробелам."""
    def header_replacer(match):
        level = len(match.group(1))
        title = match.group(2).strip()
        md_level = min(level, 6)
        return f"{'#' * md_level} {title}"

    pattern = re.compile(r'^[ \t]*(={2,6})[ \t]*(.*?)[ \t]*\1[ \t]*(?:<!--.*?-->)?[ \t]*$', re.MULTILINE)
    return pattern.sub(header_replacer, text)


def convert_lists(text: str) -> str:
    """Конвертирует списки MediaWiki в корректный Markdown."""
    lines = text.splitlines()
    res = []

    for line in lines:
        stripped = line.strip()
        # Вложенный нумерованный список
        if stripped.startswith('##'):
            content = stripped[2:].strip()
            res.append(f"    1. {content}")
        # Нумерованный список MediaWiki (# пункт или #пункт)
        elif stripped.startswith('#'):
            content = stripped[1:].strip()
            res.append(f"1. {content}")
        # Вложенный маркированный список
        elif stripped.startswith('**'):
            content = stripped[2:].strip()
            res.append(f"  * {content}")
        # Определение ; Термин : Описание
        elif stripped.startswith(';') and ':' in stripped:
            term, desc = stripped[1:].split(':', 1)
            res.append(f"**{term.strip()}**: {desc.strip()}")
        # Отступ через двоеточие (ответ/цитата)
        elif stripped.startswith(':'):
            content = stripped[1:].strip()
            res.append(f"> {content}")
        else:
            res.append(line)

    return '\n'.join(res)


def clean_neolurk_wikitext(title: str, text: str) -> str:
    """
    Полная очистка викитекста статьи Неолурка и конвертация в чистый Markdown.
    """
    if not text:
        return f"# {title}\n"

    # 0. Нормализация переводов строк и пробелов
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    text = text.replace('\xa0', ' ')

    # 1. Отрезаем служебные разделы в конце (Примечания, Ссылки, См. также и т.д.)
    text = re.split(
        r'\n==+\s*(?:Примечания|Ссылки|См\.\s*также|Источники|Литература|Галерея)\s*==+.*',
        text,
        flags=re.IGNORECASE | re.DOTALL
    )[0]

    # 2. Удаляем комментарии <!-- ... -->
    text = re.sub(r'<!--.*?-->', '', text, flags=re.DOTALL)

    # 3. Удаляем сноски <ref>...</ref> и одиночные <ref ... />
    text = re.sub(r'<ref[^>]*>.*?</ref>', '', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'<ref[^>]*/>', '', text, flags=re.IGNORECASE)

    # 4. Удаляем блоки галерей <gallery>...</gallery>
    text = re.sub(r'<gallery[^>]*>.*?</gallery>', '', text, flags=re.DOTALL | re.IGNORECASE)

    # 5. Удаляем категории [[Категория:...]] и [[Category:...]]
    text = re.sub(r'\[\[(?:Категория|Category):[^\]]+\]\]\n?', '', text, flags=re.IGNORECASE)

    # 6. Удаляем файлы и изображения с учетом вложенных скобок
    text = remove_wiki_images(text)

    # 7. Очистка таблиц {| ... |} с извлечением цитат и текста
    text = clean_tables(text)

    # 8. Раскрытие шаблонов от внутренних к внешним
    text = unwind_templates(text)

    # 9. Внешние ссылки [http://site.com Описание] -> [Описание](http://site.com)
    text = re.sub(r'\[(https?://[^\s\]]+)\s+([^\]]+)\]', r'[\2](\1)', text)
    text = re.sub(r'\[(https?://[^\s\]]+)\]', r'\1', text)

    # 10. Внутренние вики-ссылки [[Статья|Текст]] -> Текст
    def link_replacer(match):
        link_content = match.group(1).strip()
        if '|' in link_content:
            return link_content.split('|')[-1].strip()
        if link_content.startswith('#'):
            return link_content[1:].strip()
        return link_content

    text = re.sub(r'\[\[([^\]]+)\]\]', link_replacer, text)

    # 11. Базовые HTML-теги форматирования в Markdown
    text = re.sub(r'<(?:s|strike|del)>(.*?)</(?:s|strike|del)>', r'~~\1~~', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'<(?:b|strong)>(.*?)</(?:b|strong)>', r'**\1**', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'<(?:i|em)>(.*?)</(?:i|em)>', r'*\1*', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'<code>(.*?)</code>', r'`\1`', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'<br\s*/?>', '\n', text, flags=re.IGNORECASE)
    text = re.sub(r'<hr\s*/?>', '\n---\n', text, flags=re.IGNORECASE)
    text = re.sub(r'<[^>]+>', '', text)

    # Декодирование HTML-сущностей (&quot;, &laquo;, &nbsp; и т.д.)
    text = html.unescape(text)

    # 12. Конвертируем списки MediaWiki (# -> 1., ** -> *) до заголовков!
    text = convert_lists(text)

    # 13. Конвертируем заголовки в Markdown (== -> ##)
    text = convert_headers(text)

    # 14. Конвертируем жирный шрифт и курсив MediaWiki
    text = re.sub(r"'''''(.*?)'''''", r"***\1***", text)
    text = re.sub(r"'''(.*?)'''", r"**\1**", text)
    text = re.sub(r"''(.*?)''", r"*\1*", text)

    # 15. Чистка оставшихся мусорных скобок и лишних пустых строк
    text = re.sub(r'\]\]+', '', text)
    text = re.sub(r'\}\}+', '', text)
    text = re.sub(r'[ \t]+$', '', text, flags=re.MULTILINE)
    text = re.sub(r'\n{3,}', '\n\n', text).strip()

    return f"# {title}\n\n{text}"




In [12]:
"""
Модуль для обогащения статей Неолурка метаданными о синонимах (редиректах).

ВАЖНО: Исходный файл базы данных (neolurk.db) НЕ МОДИФИЦИРУЕТСЯ (доступ строго в режиме Read-Only).
Все связи между редиректами и основными статьями строятся в оперативной памяти за < 1 секунды.

Возможности:
1. Анализ всех 85 000+ редиректов и разрешение цепочек перенаправлений (A -> B -> C).
2. Очистка якорей разделов (#Раздел) и нормализация заголовков.
3. Обогащение каждой статьи полем synonyms (список всех синонимов / альтернативных названий).
4. Поиск статьи как по основному названию, так и по любому из её синонимов.
5. Потоковый итератор по всем статьям для выгрузки в JSON / векторные базы без утечек памяти.
"""



import sqlite3
import time
import re
import sys
import argparse
from collections import defaultdict
from dataclasses import dataclass
from typing import List, Dict, Optional, Iterator

# Импортируем очиститель викитекста из parsing.py
try:
    from parsing import clean_neolurk_wikitext
except ImportError:
    def clean_neolurk_wikitext(title: str, text: str) -> str:
        return text

# Корректное отображение русских символов в Windows
if sys.platform == "win32":
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")


def normalize_title(title: str) -> str:
    """Нормализует заголовок MediaWiki (заменяет _ на пробелы, удаляет невидимые символы)."""
    if not title:
        return ""
    # Удаляем невидимые символы разметки LTR/RTL (u200e и др.) и zero-width
    t = re.sub(r'[\u200e\u200f\u200b\u200c\u200d\ufeff]', '', title)
    t = t.replace("_", " ").strip()
    return t[:1].upper() + t[1:] if t else ""


def clean_redirect_target(raw_target: str) -> str:
    """Удаляет якоря секций (#...) и мусор из ссылки редиректа."""
    if not raw_target:
        return ""
    target = raw_target.split("#")[0].strip()
    return normalize_title(target)


@dataclass
class EnrichedArticle:
    id: int
    title: str
    url: str
    content: str
    synonyms: List[str]
    timestamp: str
    size: int

    @property
    def synonym_count(self) -> int:
        return len(self.synonyms)

    def to_dict(self) -> dict:
        return {
            "id": self.id,
            "title": self.title,
            "url": self.url,
            "synonyms": self.synonyms,
            "synonym_count": self.synonym_count,
            "timestamp": self.timestamp,
            "size": self.size,
            "content": self.content
        }


class NeolurkArchive:
    """
    Интерфейс для доступа к базе Неолурка в режиме Read-Only
    с автоматическим обогащением статей синонимами.
    """
    def __init__(self, db_path: str = "/kaggle/input/datasets/artemglobuzz/sigma/neolurk.db"):
        self.db_path = db_path
        # Открываем SQLite строго в режиме read-only (гарантия неизменности файла)
        self.conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
        self.redir_pattern = re.compile(r'#(?:REDIRECT|перенаправление)\s*\[\[([^\]#|]+)', re.IGNORECASE)

        self.target_to_synonyms: Dict[str, List[str]] = defaultdict(list)
        self.synonym_to_target: Dict[str, str] = {}
        self._build_synonyms_index()

    def _build_synonyms_index(self):
        """Строит граф редиректов в памяти и разрешает многошаговые цепочки."""
        t0 = time.time()
        cur = self.conn.cursor()

        # Быстрый индексированный запрос
        cur.execute("SELECT title, redirect_target, content FROM pages WHERE is_redirect = 1")
        rows = cur.fetchall()

        direct_redirects = {}
        for title, target, content in rows:
            norm_title = normalize_title(title)
            final_target = None
            if target and target.strip():
                final_target = clean_redirect_target(target)
            elif content:
                m = self.redir_pattern.search(content)
                if m:
                    final_target = clean_redirect_target(m.group(1))

            if norm_title and final_target and norm_title != final_target:
                direct_redirects[norm_title] = final_target

        # Разрешаем цепочки перенаправлений (если A -> B, а B -> C, то и A, и B ведут на C)
        for synonym, target in direct_redirects.items():
            curr = target
            visited = {synonym}
            for _ in range(5):  # Ограничение глубины для защиты от циклов
                if curr in direct_redirects and curr not in visited:
                    visited.add(curr)
                    curr = direct_redirects[curr]
                else:
                    break

            canonical_target = curr
            self.synonym_to_target[synonym] = canonical_target
            self.target_to_synonyms[canonical_target].append(synonym)

        # Сортируем списки синонимов и удаляем дубликаты
        for k in self.target_to_synonyms:
            self.target_to_synonyms[k] = sorted(list(set(self.target_to_synonyms[k])))

        t1 = time.time()
        self.index_build_time = t1 - t0
        self.total_redirects_count = len(direct_redirects)
        self.enriched_targets_count = len(self.target_to_synonyms)

    def get_synonyms_for(self, title: str) -> List[str]:
        """Возвращает список всех синонимов для канонической статьи."""
        norm = normalize_title(title)
        canonical = self.synonym_to_target.get(norm, norm)
        return self.target_to_synonyms.get(canonical, [])

    def resolve_canonical_title(self, title_or_synonym: str) -> str:
        """По синониму возвращает настоящее каноническое название статьи."""
        norm = normalize_title(title_or_synonym)
        return self.synonym_to_target.get(norm, norm)

    def get_article(self, title_or_synonym: str, clean_markdown: bool = False) -> Optional[EnrichedArticle]:
        """
        Получает статью по её названию ИЛИ по любому из её синонимов.
        Если clean_markdown=True, поле content возвращается уже очищенным в формате Markdown.
        """
        canonical_title = self.resolve_canonical_title(title_or_synonym)

        cur = self.conn.cursor()
        cur.execute("""
            SELECT id, title, url, content, timestamp, size
            FROM pages
            WHERE is_redirect = 0 AND (title = ? OR title = ?)
            LIMIT 1
        """, (canonical_title, canonical_title.replace(" ", "_")))
        row = cur.fetchone()

        if not row:
            return None

        pid, title, url, raw_content, ts, size = row
        synonyms = self.get_synonyms_for(title)

        content = clean_neolurk_wikitext(title, raw_content) if clean_markdown else raw_content

        return EnrichedArticle(
            id=pid,
            title=title,
            url=url,
            content=content,
            synonyms=synonyms,
            timestamp=ts,
            size=size
        )

    def iter_articles(
        self,
        limit: Optional[int] = None,
        clean_markdown: bool = False,
        start_id: int = 0
    ) -> Iterator[EnrichedArticle]:
        """
        Потоковый генератор по всем полноценным статьям базы,
        обогащённым списком синонимов на лету.
        Поддерживает дозапись и продолжение с start_id.
        """
        cur = self.conn.cursor()
        query = "SELECT id, title, url, content, timestamp, size FROM pages WHERE is_redirect = 0 AND id > ? ORDER BY id ASC"
        if limit:
            query += f" LIMIT {limit}"

        cur.execute(query, (start_id,))
        while True:
            batch = cur.fetchmany(1000)
            if not batch:
                break
            for pid, title, url, raw_content, ts, size in batch:
                synonyms = self.get_synonyms_for(title)
                content = clean_neolurk_wikitext(title, raw_content) if clean_markdown else raw_content
                yield EnrichedArticle(
                    id=pid,
                    title=title,
                    url=url,
                    content=content,
                    synonyms=synonyms,
                    timestamp=ts,
                    size=size
                )


In [13]:
"""
Модуль для RAG-чанкинга статей Неолурка на базе LangChain.

Архитектура: Section-Aware Parent-Child (Иерархический чанкинг по разделам).

Почему Parent-Child — лучший выбор для энциклопедических данных:
1. Решает проблему размытия эмбеддингов: поиск идёт по компактным Child-чанкам (300-500 символов),
   что даёт высокую точность совпадения цитат, сленга и фактов.
2. Решает проблему нехватки контекста у LLM: найденный Child-чанк ссылается на Parent-документ
   (весь логический раздел целиком, 1500-2500 символов), который и передаётся в языковую модель.
3. Учитывает структуру Вики: чанки привязаны к заголовкам Markdown (## Раздел), предотвращая
   разрыв предложений и склейку несвязанных тем.
4. Контекстный префикс (Contextual Retrieval): каждый дочерний чанк обогащается строкой
   с каноническим названием статьи, списком синонимов и текущим разделом.
"""

import sys
import json
import argparse
from typing import List, Tuple, Dict, Any, Optional, Iterator

# LangChain компоненты
from langchain_core.documents import Document
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter
)

# Модуль доступа к базе с синонимами




class NeolurkHierarchicalChunker:
    """
    Иерархический чанкер для RAG.

    Делит каждую статью на:
    - Parent Documents (для генерации LLM): логические разделы Markdown.
    - Child Chunks (для векторного поиска): небольшие смысловые фрагменты с метаданными.
    """
    def __init__(
        self,
        parent_chunk_size: int = 3000,
        parent_chunk_overlap: int = 200,
        child_chunk_size: int = 750,
        child_chunk_overlap: int = 70,
        add_context_prefix_to_child: bool = True
    ):
        self.parent_chunk_size = parent_chunk_size
        self.parent_chunk_overlap = parent_chunk_overlap
        self.child_chunk_size = child_chunk_size
        self.child_chunk_overlap = child_chunk_overlap
        self.add_context_prefix_to_child = add_context_prefix_to_child

        # Сплиттер заголовков Markdown
        self.markdown_splitter = MarkdownHeaderTextSplitter(
            headers_to_split_on=[
                ("#", "h1"),
                ("##", "h2"),
                ("###", "h3"),
                ("####", "h4"),
            ],
            strip_headers=False
        )

        # Сплиттер для разбивки слишком длинных разделов на родительские блоки
        self.parent_text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=parent_chunk_size,
            chunk_overlap=parent_chunk_overlap,
            separators=["\n\n", "\n", ". ", " ", ""]
        )

        # Сплиттер для дочерних поисковых чанков
        self.child_text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=child_chunk_size,
            chunk_overlap=child_chunk_overlap,
            separators=["\n\n", "\n", ". ", "; ", ", ", " ", ""]
        )

    def chunk_article(self, article: EnrichedArticle) -> Tuple[List[Document], List[Document]]:
        """
        Разбивает одну статью на списки (parents, children).
        """
        parent_docs: List[Document] = []
        child_docs: List[Document] = []

        synonyms_str = ", ".join(article.synonyms) if article.synonyms else ""

        # Базовые метаданные статьи (для фильтрации, цитирования и поиска)
        base_meta = {
            "doc_id": article.id,
            "title": article.title,
            "url": article.url,
            "synonyms": article.synonyms,
            "synonyms_str": synonyms_str,
            "synonym_count": article.synonym_count,
            "timestamp": article.timestamp,
        }

        # 1. Деление по логическим секциям Markdown (## Раздел)
        raw_sections = self.markdown_splitter.split_text(article.content)
        if not raw_sections:
            raw_sections = [Document(page_content=article.content, metadata={})]

        parent_idx = 0

        for sec in raw_sections:
            # Формируем путь раздела: Статья > Раздел > Подраздел
            h_parts = [sec.metadata.get(f"h{lvl}") for lvl in (1, 2, 3, 4) if sec.metadata.get(f"h{lvl}")]
            section_breadcrumb = " > ".join(h_parts) if h_parts else article.title

            sec_content = sec.page_content.strip()
            if not sec_content:
                continue

            # 2. Формирование Parent документов
            if len(sec_content) <= self.parent_chunk_size:
                parent_texts = [sec_content]
            else:
                parent_texts = self.parent_text_splitter.split_text(sec_content)

            for p_text in parent_texts:
                parent_idx += 1
                parent_id = f"art_{article.id}_p{parent_idx}"

                parent_metadata = {
                    **base_meta,
                    "parent_id": parent_id,
                    "section": section_breadcrumb,
                    "chunk_type": "parent",
                    "char_count": len(p_text),
                }

                parent_doc = Document(
                    page_content=p_text,
                    metadata=parent_metadata
                )
                parent_docs.append(parent_doc)

                # 3. Формирование Child документов из текущего Parent
                child_texts = self.child_text_splitter.split_text(p_text)
                for child_idx, c_text in enumerate(child_texts, 1):
                    child_id = f"{parent_id}_c{child_idx}"

                    # Contextual Retrieval prefix:
                    # Добавляем контекст статьи, чтобы изолированный абзац легко находился по эмбеддингам
                    if self.add_context_prefix_to_child:
                        context_header = f"[Статья: {article.title}"
                        if synonyms_str:
                            context_header += f" | Синонимы: {synonyms_str}"
                        context_header += f" | Раздел: {section_breadcrumb}]\n"
                        child_page_content = context_header + c_text
                    else:
                        child_page_content = c_text

                    child_metadata = {
                        **base_meta,
                        "chunk_id": child_id,
                        "parent_id": parent_id,
                        "section": section_breadcrumb,
                        "chunk_type": "child",
                        "child_index": child_idx,
                        "char_count": len(c_text),
                    }

                    child_doc = Document(
                        page_content=child_page_content,
                        metadata=child_metadata
                    )
                    child_docs.append(child_doc)

        return parent_docs, child_docs

    def chunk_articles_stream(
        self,
        articles: Iterator[EnrichedArticle]
    ) -> Iterator[Tuple[List[Document], List[Document]]]:
        """Потоковая обработка статей."""
        for art in articles:
            yield self.chunk_article(art)


In [32]:
"""
Единый модуль RAG для Неолурка:
1. Hybrid Retrieval: Dense (FAISS) + Lexical (BM25) со слиянием через Reciprocal Rank Fusion (RRF).
2. Reranker: Cross-Encoder (BAAI/bge-reranker-base) с порогом отсечения релевантности (score_threshold).
   Если порог не набран — система честно сообщает, что не знает ответа.
3. Parent-Child: поиск по гранулярным Child-чанкам, передача в LLM цельного Parent-раздела.
4. Генерация: локальная LLM через Ollama (по умолчанию Qwen 2.5: qwen2.5:7b).
5. Модели эмбеддингов с поддержкой русского языка:
   - 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2' (по умолчанию: лёгкая, 384d, ~470 МБ)
   - 'intfloat/multilingual-e5-base' (768d)
   - 'deepvk/USER-bge-m3' (1024d)
   - 'cointegrated/rubert-tiny2' (312d, сверхлёгкая)
"""

import os
import sys
import re
import gc
import json
import time
import math
import pickle
import argparse
from typing import List, Dict, Any, Optional, Tuple

import faiss
import numpy as np
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.docstore.in_memory import InMemoryDocstore

try:
    import bm25s
except ImportError:
    bm25s = None

try:
    from langchain_ollama import ChatOllama
except ImportError:
    ChatOllama = None

# Модели
RUSSIAN_EMBEDDING_MODELS = {
    "minilm": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "e5": "intfloat/multilingual-e5-base",
    "bge-m3": "deepvk/USER-bge-m3",
    "tiny2": "cointegrated/rubert-tiny2",
    "giga": "ai-sage/Giga-Embeddings-instruct-480M-0826",
}

RUSSIAN_RERANKER_MODELS = {
    "dity": "DiTy/cross-encoder-russian-msmarco",   # Сверхлёгкий, быстрый, специально для русского языка (~117 МБ)
    "bge": "BAAI/bge-reranker-base",                 # Мультиязычный бейзлайн (~1.1 ГБ)
}

DEFAULT_EMBED_MODEL = RUSSIAN_EMBEDDING_MODELS["minilm"]
DEFAULT_RERANKER_MODEL = RUSSIAN_RERANKER_MODELS["dity"]

RUSSIAN_LLM_MODELS = {
    "qwen-3b": "Qwen/Qwen2.5-3B-Instruct",      # Быстрый, отлично знает русский, ~6 ГБ VRAM в FP16
    "qwen-1.5b": "Qwen/Qwen2.5-1.5B-Instruct",  # Сверхлёгкий, ~3 ГБ VRAM
    "qwen-7b": "Qwen/Qwen2.5-7B-Instruct",      # Максимальное качество, ~14 ГБ VRAM
}
DEFAULT_LLM_MODEL = RUSSIAN_LLM_MODELS["qwen-3b"]
DEFAULT_OLLAMA_MODEL = "qwen2.5:7b"
DEFAULT_INDEX_DIR = "/kaggle/input/datasets/artemglobuzz/sigma-papka/new_pr"



def normalize_score(s: float) -> float:
    """
    Нормализация скора реранкера в диапазон [0.0, 1.0].
    Если модель уже возвращает вероятности (например, DiTy cross-encoder с sigmoid),
    значение сохраняется как есть. Если возвращаются сырые логиты (например, BGE),
    применяется сигмоида.
    """
    if s < 0.0 or s > 1.0:
        try:
            return 1.0 / (1.0 + math.exp(-s))
        except OverflowError:
            return 0.0 if s < 0 else 1.0
    return float(s)


def tokenize_ru(text: str) -> List[str]:
    """Лёгкий токенизатор для русского и английского текста (для BM25)."""
    return re.findall(r"[a-zA-Zа-яА-Я0-9ёЁ]+", text.lower())


def get_russian_stemmer():
    """Стеммер для русского языка для точного лексического поиска по словоформам."""
    try:
        from nltk.stem.snowball import SnowballStemmer
        stem = SnowballStemmer("russian").stem
        return lambda words: [stem(w) for w in words]
    except Exception:
        def _light_stem(w):
            return re.sub(
                r"(ами|ями|ов|ев|ей|ам|ям|ом|ем|ой|ей|ия|ии|ие|ий|ью|ья|ье|ый|ой|ий|ая|ое|ые|ых|их|ую|юю|а|е|и|о|у|ы|ь|я|ю)$",
                "",
                w.lower()
            )
        return lambda words: [_light_stem(w) for w in words]


class BM25SIndex:
    """Высокопроизводительный лексический ретривер на базе C/NumPy (bm25s)."""
    def __init__(self, retriever: Any, corpus: List[Dict[str, Any]]):
        self.retriever = retriever
        self.corpus = corpus
        self.stemmer = get_russian_stemmer()

    def search(self, query: str, k: int = 10) -> List[Document]:
        """Поиск по запросу с русской токенизацией и стеммингом (время отклика ~2-5 мс)."""
        if bm25s is None:
            return []
        q_tokens = bm25s.tokenize([query], stopwords="ru", stemmer=self.stemmer, show_progress=False)
        results, scores = self.retriever.retrieve(
            q_tokens,
            corpus=self.corpus,
            k=min(k, len(self.corpus)),
            show_progress=False
        )
        docs = []
        for doc_item, score in zip(results[0], scores[0]):
            if score <= 0:
                continue
            meta = dict(doc_item.get("metadata", {}))
            meta["bm25_score"] = float(score)
            docs.append(Document(
                page_content=doc_item.get("page_content", ""),
                metadata=meta
            ))
        return docs

    def save(self, folder_path: str):
        os.makedirs(folder_path, exist_ok=True)
        self.retriever.save(folder_path, corpus=self.corpus)

    @classmethod
    def load(cls, folder_path: str) -> "BM25SIndex":
        if bm25s is None:
            raise ImportError("Пакет 'bm25s' не установлен. Установите: pip install bm25s")
        retriever = bm25s.BM25.load(folder_path, load_corpus=True)
        return cls(retriever=retriever, corpus=retriever.corpus)


def get_embeddings(
    model_name: str = DEFAULT_EMBED_MODEL,
    batch_size: int = 512
) -> HuggingFaceEmbeddings:
    """Инициализация модели эмбеддингов с автоматическим выбором GPU и FP16."""
    try:
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
    except (ImportError, Exception):
        device = "cpu"

    model_kwargs = {"device": device, "trust_remote_code": True}
    if device == "cuda":
        model_kwargs["model_kwargs"] = {"torch_dtype": torch.float16}

    return HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs=model_kwargs,
        encode_kwargs={"normalize_embeddings": True, "batch_size": batch_size}
    )


class Reranker:
    """Кросс-энкодер реранкер для точной оценки релевантности пар (запрос, документ)."""
    def __init__(self, model_name: str = DEFAULT_RERANKER_MODEL):
        from sentence_transformers import CrossEncoder
        try:
            import torch
            device = "cuda" if torch.cuda.is_available() else "cpu"
        except (ImportError, Exception):
            device = "cpu"
        print(f"[*] Загрузка модели реранкера: {model_name} (device: {device})...")
        self.model = CrossEncoder(model_name, device=device)

    def compute_scores(self, query: str, texts: List[str]) -> List[float]:
        """Возвращает нормализованные скоры (0.0 - 1.0) для списка текстов."""
        if not texts:
            return []
        pairs = [[query, text] for text in texts]
        raw_scores = self.model.predict(pairs)
        return [normalize_score(float(s)) for s in raw_scores]


def load_docstore(docstore_path: str) -> Dict[str, Any]:
    """
    Безопасная загрузка docstore.json с автоматическим исправлением обрыва
    в случае внезапной остановки процесса или сбоя ядра.
    """
    if not os.path.exists(docstore_path):
        raise FileNotFoundError(f"Файл {docstore_path} не найден.")
    try:
        with open(docstore_path, "r", encoding="utf-8") as f:
            return json.load(f)
    except (json.JSONDecodeError, UnicodeDecodeError) as e:
        print(f"[!] Предупреждение: {docstore_path} был прерван при записи ({e}). Выполняем авто-восстановление...")
        with open(docstore_path, "rb") as f:
            raw_bytes = f.read()
        raw_text = raw_bytes.decode("utf-8", errors="ignore").rstrip()
        last_brace = raw_text.rfind("}}")
        if last_brace != -1:
            raw_text = raw_text[:last_brace + 2] + "\n}"
        else:
            last_single = raw_text.rfind("}")
            raw_text = raw_text[:last_single + 1] + "\n}"
        docstore = json.loads(raw_text)
        print(f"[✔] docstore.json успешно восстановлен! Загружено разделов: {len(docstore):,}")
        tmp_fixed = docstore_path + ".tmp"
        with open(tmp_fixed, "w", encoding="utf-8") as f:
            json.dump(docstore, f, ensure_ascii=False)
        os.replace(tmp_fixed, docstore_path)
        return docstore


class HybridNeolurkRetriever:
    """
    Гибридный ретривер (Dense FAISS + Lexical BM25 + RRF)
    с опциональным Cross-Encoder реранкингом и порогом релевантности.
    Работает как в гибридном режиме (FAISS + BM25), так и в чисто плотном (только FAISS).
    """
    def __init__(
        self,
        vectorstore: FAISS,
        bm25: Optional[BM25Retriever],
        docstore: Dict[str, Dict[str, Any]],
        model_name: str = DEFAULT_EMBED_MODEL
    ):
        self.vectorstore = vectorstore
        self.bm25 = bm25
        self.docstore = docstore
        self.model_name = model_name
        self._reranker: Optional[Reranker] = None

    def get_reranker(self, model_name: str = DEFAULT_RERANKER_MODEL) -> Reranker:
        """Ленивая загрузка реранкера (только когда он требуется)."""
        if self._reranker is None:
            self._reranker = Reranker(model_name)
        return self._reranker

    def save(self, folder_path: str = DEFAULT_INDEX_DIR):
        os.makedirs(folder_path, exist_ok=True)
        self.vectorstore.save_local(folder_path)
        if self.bm25 is not None:
            if hasattr(self.bm25, "save"):
                self.bm25.save(os.path.join(folder_path, "bm25s"))
            else:
                try:
                    with open(os.path.join(folder_path, "bm25.pkl"), "wb") as f:
                        pickle.dump(self.bm25, f)
                except Exception as e:
                    print(f"[!] Не удалось сериализовать legacy BM25 ({e})")
        tmp_docstore = os.path.join(folder_path, "docstore.json.tmp")
        docstore_file = os.path.join(folder_path, "docstore.json")
        with open(tmp_docstore, "w", encoding="utf-8") as f:
            json.dump(self.docstore, f, ensure_ascii=False)
        os.replace(tmp_docstore, docstore_file)
        print(f"[✔] Индекс успешно сохранён в '{folder_path}'")


    @classmethod
    def load(
        cls,
        folder_path: str = DEFAULT_INDEX_DIR,
        embeddings: Optional[HuggingFaceEmbeddings] = None
    ) -> "HybridNeolurkRetriever":
        chk_path = os.path.join(folder_path, "checkpoint.json")
        model_name = DEFAULT_EMBED_MODEL
        if os.path.exists(chk_path):
            try:
                with open(chk_path, "r", encoding="utf-8") as f:
                    chk = json.load(f)
                model_name = chk.get("model_name", DEFAULT_EMBED_MODEL)
                print(f"[*] Загрузка индекса из '{folder_path}':")
                print(f"    - Модель эмбеддингов:  {model_name}")
                print(f"    - Статей в индексе:    {chk.get('articles_processed', 'n/a')} (последний ID: {chk.get('last_page_id', 'n/a')})")
                print(f"    - Векторов в FAISS:    {chk.get('faiss_total', 'n/a')}")
                print(f"    - Статус индексации:   {chk.get('status', 'in_progress')}")
            except Exception:
                pass

        if embeddings is None:
            embeddings = get_embeddings(model_name)

        vectorstore = FAISS.load_local(
            folder_path=folder_path,
            embeddings=embeddings,
            allow_dangerous_deserialization=True
        )

        bm25 = None
        bm25s_dir = os.path.join(folder_path, "bm25s")
        bm25_file = os.path.join(folder_path, "bm25.pkl")
        if bm25s is not None and os.path.exists(bm25s_dir):
            try:
                bm25 = BM25SIndex.load(bm25s_dir)
                print(f"    - Лексический поиск:   BM25S (C/NumPy, быстрый)")
            except Exception as e:
                print(f"[!] Ошибка загрузки BM25S: {e}")
        elif os.path.exists(bm25_file):
            try:
                with open(bm25_file, "rb") as f:
                    bm25 = pickle.load(f)
                print(f"    - Лексический поиск:   rank_bm25 (legacy pkl)")
            except Exception as e:
                print(f"[!] Ошибка загрузки BM25: {e}")

        docstore = load_docstore(os.path.join(folder_path, "docstore.json"))

        return cls(vectorstore=vectorstore, bm25=bm25, docstore=docstore, model_name=model_name)


    def search_candidates(
        self,
        query: str,
        k_dense: int = 30,
        k_bm25: int = 30,
        rrf_k: int = 60
    ) -> List[Document]:
        """Первичный отбор кандидатов (Dense FAISS + Lexical BM25 со слиянием через RRF)."""
        dense_query = query
        if "giga" in self.model_name.lower():
            dense_query = f"Instruct: Дан вопрос, найди подходящий фрагмент статьи из энциклопедии Неолурк.\nQuery: {query}"
        elif "e5" in self.model_name.lower():
            dense_query = f"query: {query}"

        dense_results = self.vectorstore.similarity_search(dense_query, k=k_dense)

        # Если BM25 ещё не построен, возвращаем результаты dense поиска
        if self.bm25 is None:
            return dense_results

        if hasattr(self.bm25, "search"):
            bm25_results = self.bm25.search(query, k=k_bm25)
        elif hasattr(self.bm25, "invoke"):
            self.bm25.k = k_bm25
            bm25_results = self.bm25.invoke(query)
        else:
            bm25_results = []

        rrf_scores: Dict[str, float] = {}
        candidate_map: Dict[str, Document] = {}

        for rank, doc in enumerate(dense_results, start=1):
            pid = doc.metadata.get("parent_id", doc.metadata.get("chunk_id", str(id(doc))))
            if pid not in candidate_map:
                candidate_map[pid] = doc
            rrf_scores[pid] = rrf_scores.get(pid, 0.0) + (0.5 / (rrf_k + rank))

        for rank, doc in enumerate(bm25_results, start=1):
            pid = doc.metadata.get("parent_id", doc.metadata.get("chunk_id", str(id(doc))))
            if pid not in candidate_map:
                candidate_map[pid] = doc
            rrf_scores[pid] = rrf_scores.get(pid, 0.0) + (0.5 / (rrf_k + rank))

        sorted_pids = sorted(rrf_scores.keys(), key=lambda pid: rrf_scores[pid], reverse=True)
        return [candidate_map[pid] for pid in sorted_pids]

    def search_and_rerank(
        self,
        query: str,
        top_parents: int = 3,
        k_candidates: int = 20,
        score_threshold: float = 0.3,
        rerank: bool = True,
        reranker_model: str = DEFAULT_RERANKER_MODEL
    ) -> Tuple[List[Document], bool]:
        """
        Полный цикл поиска:
        1. Гибридный отбор топ кандидатов (FAISS + BM25).
        2. Реранкинг через Cross-Encoder.
        3. Проверка score_threshold: если скор ниже порога, возвращает ([], False).
        4. Разрешение Child -> Parent (цельный раздел для LLM без дубликатов).

        Возвращает: (список_parent_документов, пройден_ли_порог)
        """
        candidates = self.search_candidates(query, k_dense=k_candidates, k_bm25=k_candidates)
        if not candidates:
            return [], False

        # Если реранкинг включен
        if rerank:
            reranker = self.get_reranker(reranker_model)
            texts = [c.page_content for c in candidates]
            scores = reranker.compute_scores(query, texts)

            # Привязываем скор реранкера к чанкам
            scored_candidates = list(zip(candidates, scores))
            # Сортируем по убыванию скора реранкера
            scored_candidates.sort(key=lambda x: x[1], reverse=True)

            # Проверяем лучший скор на соответствие порогу
            best_score = scored_candidates[0][1] if scored_candidates else 0.0
            if best_score < score_threshold:
                # Порог не пройден — релевантной информации нет
                return [], False

            # Оставляем только те, что преодолели порог
            filtered_candidates = [
                (doc, score) for doc, score in scored_candidates if score >= score_threshold
            ]
        else:
            filtered_candidates = [(doc, 1.0) for doc in candidates]

        # Извлечение уникальных Parent-документов
        seen_parents = set()
        parent_documents: List[Document] = []

        for child, score in filtered_candidates:
            p_id = child.metadata.get("parent_id")
            if not p_id or p_id in seen_parents:
                continue

            p_data = self.docstore.get(p_id)
            if p_data:
                seen_parents.add(p_id)
                meta = dict(p_data["metadata"])
                meta["rerank_score"] = round(score, 4)
                meta["matched_child_id"] = child.metadata.get("chunk_id")
                parent_documents.append(Document(page_content=p_data["page_content"], metadata=meta))

                if len(parent_documents) >= top_parents:
                    break

        return parent_documents, True


def build_hybrid_index(
    db_path: str = "/content/neolurk.db",
    output_dir: str = DEFAULT_INDEX_DIR,
    limit: Optional[int] = None,
    model_name: str = DEFAULT_EMBED_MODEL,
    embeddings: Optional[Any] = None,
    batch_size: int = 512,
    save_interval: int = 1000,
    reset: bool = False
) -> HybridNeolurkRetriever:
    """
    Шаг 1: Построение Dense Vector Store (чистый FAISS Flat) + docstore.json с чекпоинтами.
    Чанки добавляются в FAISS батчами — потребление RAM стабильно ~1.5 ГБ (без OOM!).
    """
    t0 = time.time()
    os.makedirs(output_dir, exist_ok=True)
    checkpoint_file = os.path.join(output_dir, "checkpoint.json")
    docstore_file = os.path.join(output_dir, "docstore.json")

    if embeddings is None:
        embeddings = get_embeddings(model_name, batch_size=batch_size)
    vectorstore: Optional[FAISS] = None
    docstore: Dict[str, Dict[str, Any]] = {}
    start_id = 0
    total_articles_indexed = 0

    # Проверка существующего чекпоинта
    if not reset and os.path.exists(checkpoint_file):
        try:
            with open(checkpoint_file, "r", encoding="utf-8") as f:
                chk = json.load(f)
            start_id = chk.get("last_page_id", 0)
            total_articles_indexed = chk.get("articles_processed", 0)
            print(f"[+] ОБНАРУЖЕН ЧЕКПОИНТ: продолжение с ID статьи {start_id}")
            print(f"    - Уже обработано статей: {total_articles_indexed}")

            if os.path.exists(os.path.join(output_dir, "index.faiss")):
                vectorstore = FAISS.load_local(output_dir, embeddings, allow_dangerous_deserialization=True)
                print(f"    - Загружено векторов:    {vectorstore.index.ntotal}")

            if os.path.exists(docstore_file):
                with open(docstore_file, "r", encoding="utf-8") as f:
                    docstore = json.load(f)
                print(f"    - Загружено разделов:    {len(docstore)}")
        except Exception as e:
            print(f"[!] Ошибка чтения чекпоинта ({e}), начинаем сборку заново.")
            start_id = 0
            total_articles_indexed = 0
            vectorstore = None
            docstore = {}
    elif reset:
        print(f"[*] Флаг --reset: сборка индекса с нуля...")

    print(f"[*] Открытие базы: {db_path} (Read-Only)")
    archive = NeolurkArchive(db_path)
    chunker = NeolurkHierarchicalChunker()

    session_articles = 0
    batch: List[Document] = []
    current_last_id = start_id

    def commit_batch():
        """Сбрасывает накопленный батч чанков в FAISS."""
        nonlocal vectorstore
        if not batch:
            return
        if vectorstore is None:
            vectorstore = FAISS.from_documents(batch, embeddings)
        else:
            vectorstore.add_documents(batch)
        batch.clear()

    def save_checkpoint(last_id: int, is_final: bool = False):
        """Сохраняет текущий прогресс на диск с атомарной защитой от повреждения."""
        commit_batch()
        if vectorstore is not None:
            tmp_save_dir = os.path.join(output_dir, "_tmp_save")
            vectorstore.save_local(tmp_save_dir)
            for fname in os.listdir(tmp_save_dir):
                src = os.path.join(tmp_save_dir, fname)
                dst = os.path.join(output_dir, fname)
                os.replace(src, dst)
            if os.path.exists(tmp_save_dir):
                try:
                    os.rmdir(tmp_save_dir)
                except OSError:
                    pass

        tmp_docstore = docstore_file + ".tmp"
        with open(tmp_docstore, "w", encoding="utf-8") as f:
            json.dump(docstore, f, ensure_ascii=False)
        os.replace(tmp_docstore, docstore_file)

        chk_data = {
            "last_page_id": last_id,
            "articles_processed": total_articles_indexed + session_articles,
            "parents_count": len(docstore),
            "faiss_total": vectorstore.index.ntotal if vectorstore else 0,
            "model_name": model_name,
            "status": "completed" if is_final else "in_progress",
            "updated_at": time.strftime("%Y-%m-%d %H:%M:%S")
        }
        tmp_chk = checkpoint_file + ".tmp"
        with open(tmp_chk, "w", encoding="utf-8") as f:
            json.dump(chk_data, f, ensure_ascii=False, indent=2)
        os.replace(tmp_chk, checkpoint_file)


        gc.collect()
        total_st = total_articles_indexed + session_articles
        vs_cnt = vectorstore.index.ntotal if vectorstore else 0
        print(f"\n[💾 Чекпоинт сохранён] Статья ID: {last_id} | Статей: {total_st} | Разделов: {len(docstore)} | Векторов FAISS: {vs_cnt}")

    limit_str = f"{limit} статей" if limit else "до конца базы"
    print(f"[*] Обработка статей (лимит в этой сессии: {limit_str})...")

    try:
        for article in archive.iter_articles(start_id=start_id, limit=limit, clean_markdown=True):
            session_articles += 1
            current_last_id = article.id
            parents, children = chunker.chunk_article(article)

            # 1. Сохраняем родительские разделы в docstore
            for p in parents:
                docstore[p.metadata["parent_id"]] = {
                    "page_content": p.page_content,
                    "metadata": p.metadata
                }

            # 2. Накапливаем чанки в батч
            batch.extend(children)

            # 3. Как только набрался батч — сразу отправляем в FAISS
            if len(batch) >= batch_size:
                commit_batch()

            # 4. Прогресс-бар
            if session_articles % 25 == 0:
                faiss_cnt = vectorstore.index.ntotal if vectorstore else 0
                sys.stdout.write(
                    f"\r[+] Сессия: {session_articles} ст. | Всего ст.: {total_articles_indexed + session_articles} "
                    f"| Разделов: {len(docstore)} | Векторов: {faiss_cnt}"
                )
                sys.stdout.flush()

            # 5. Периодическое сохранение чекпоинта
            if session_articles % save_interval == 0:
                save_checkpoint(current_last_id, is_final=False)

    except KeyboardInterrupt:
        print("\n[!] Остановка пользователем. Сохраняем аварийный чекпоинт...")
        save_checkpoint(current_last_id, is_final=False)
        print("[+] Чекпоинт сохранён! Индекс готов к работе или продолжению сборки.")
        bm25 = None
        bm25s_dir = os.path.join(output_dir, "bm25s")
        bm25_file = os.path.join(output_dir, "bm25.pkl")
        if bm25s is not None and os.path.exists(bm25s_dir):
            try:
                bm25 = BM25SIndex.load(bm25s_dir)
            except Exception:
                pass
        elif os.path.exists(bm25_file):
            try:
                with open(bm25_file, "rb") as f:
                    bm25 = pickle.load(f)
            except Exception:
                pass
        return HybridNeolurkRetriever(vectorstore=vectorstore, bm25=bm25, docstore=docstore, model_name=model_name)

    # Финальное сохранение сессии
    save_checkpoint(current_last_id, is_final=True)
    print(f"\n[✔] Векторная база FAISS успешно построена за {time.time() - t0:.1f}с.")
    print(f"    - Обработано статей в сессии: {session_articles}")
    print(f"    - Всего статей в базе:        {total_articles_indexed + session_articles}")
    print(f"    - Всего разделов (Parent):    {len(docstore)}")
    print(f"    - Всего векторов в FAISS:     {vectorstore.index.ntotal}")

    bm25 = None
    bm25s_dir = os.path.join(output_dir, "bm25s")
    bm25_file = os.path.join(output_dir, "bm25.pkl")
    if bm25s is not None and os.path.exists(bm25s_dir):
        try:
            bm25 = BM25SIndex.load(bm25s_dir)
        except Exception:
            pass
    elif os.path.exists(bm25_file):
        try:
            with open(bm25_file, "rb") as f:
                bm25 = pickle.load(f)
        except Exception:
            pass

    return HybridNeolurkRetriever(vectorstore=vectorstore, bm25=bm25, docstore=docstore, model_name=model_name)


def build_bm25_index(
    output_dir: str = DEFAULT_INDEX_DIR,
    db_path: str = "/content/neolurk.db",
    source: str = "parent"
) -> Any:
    """
    Шаг 2: Построение сверхбыстрого индекса BM25.
    Использует библиотеку bm25s (на C/NumPy) со стеммингом для русского языка.
    Запрос выполняется за 2-5 мс, потребление RAM в 4 раза меньше.
    """
    t0 = time.time()
    bm25s_dir = os.path.join(output_dir, "bm25s")
    bm25_file = os.path.join(output_dir, "bm25.pkl")

    if source == "parent":
        docstore_file = os.path.join(output_dir, "docstore.json")
        if not os.path.exists(docstore_file):
            raise FileNotFoundError(f"Файл {docstore_file} не найден. Сначала выполните build_hybrid_index().")
        print(f"[*] Загрузка разделов из {docstore_file} для BM25...")
        docstore = load_docstore(docstore_file)

        print(f"[*] Подготовка {len(docstore)} документов для BM25...")

        corpus = []
        for p_id, p_data in docstore.items():
            meta = dict(p_data["metadata"])
            meta["parent_id"] = p_id
            meta["chunk_id"] = p_id
            corpus.append({
                "page_content": p_data["page_content"],
                "metadata": meta
            })
    else:
        print(f"[*] Потоковое чтение статей из {db_path} для child-уровня BM25...")
        archive = NeolurkArchive(db_path)
        chunker = NeolurkHierarchicalChunker()
        corpus = []
        for art in archive.iter_articles(clean_markdown=True):
            _, children = chunker.chunk_article(art)
            for c in children:
                corpus.append({
                    "page_content": c.page_content,
                    "metadata": {"chunk_id": c.metadata.get("chunk_id"), "parent_id": c.metadata.get("parent_id")}
                })
            if len(corpus) % 100000 == 0:
                print(f"[+] Собрано чанков: {len(corpus)}...")

    # Если bm25s установлен — строим C/NumPy индекс
    if bm25s is not None:
        texts = [item["page_content"] for item in corpus]
        stemmer_fn = get_russian_stemmer()
        print(f"[*] Токенизация и стемминг {len(texts)} текстов через bm25s (русский язык)...")
        tokens = bm25s.tokenize(texts, stopwords="ru", stemmer=stemmer_fn)

        print("[*] Построение разреженной матрицы BM25S...")
        retriever = bm25s.BM25()
        retriever.index(tokens)

        print(f"[*] Сохранение индекса BM25S в '{bm25s_dir}'...")
        retriever.save(bm25s_dir, corpus=corpus)
        print(f"[✔] Сверхбыстрый индекс BM25S построен и сохранён в '{bm25s_dir}' за {time.time() - t0:.1f}с!")
        return BM25SIndex(retriever=retriever, corpus=corpus)
    else:
        print("[!] Пакет 'bm25s' не найден, используем классический rank_bm25...")
        bm25_docs = [Document(page_content=c["page_content"], metadata=c["metadata"]) for c in corpus]
        bm25 = BM25Retriever.from_documents(bm25_docs, preprocess_func=tokenize_ru)
        with open(bm25_file, "wb") as f:
            pickle.dump(bm25, f)
        print(f"[✔] Индекс BM25 сохранён в '{bm25_file}' за {time.time() - t0:.1f}с!")
        return bm25



In [25]:
import torch
from transformers import pipeline

In [26]:
generator = pipeline(
        "text-generation",
        model="Qwen/Qwen2.5-3B-Instruct",
        torch_dtype=torch.float16,
        device="cuda:1"
    )
def generate_with_hf(query, docs, max_new_tokens=512):

# Загружаем модель прямо на свободную вторую карту (cuda:1)
    
    context = "\n\n---\n\n".join([f"[{d.metadata.get('title')}]\n{d.page_content}" for d in docs])
    
    messages = [
        {"role": "system", "content": "Ты ассистент по энциклопедии Неолурк. Отвечай на вопрос строго по предоставленному контексту. Если инфы нет, скажи об этом."},
        {"role": "user", "content": f"Контекст:\n{context}\n\nВопрос: {query}"}
    ]
    
    prompt = generator.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    out = generator(prompt, max_new_tokens=max_new_tokens, temperature=0.3, do_sample=True)
    return out[0]["generated_text"][len(prompt):].strip()

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [18]:

retr1 = HybridNeolurkRetriever.load("/kaggle/working/my_index")

/usr/lib/python3.12/pty.py:95: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


[*] Загрузка индекса из '/kaggle/working/my_index':
    - Модель эмбеддингов:  sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
    - Статей в индексе:    49359 (последний ID: 680733)
    - Векторов в FAISS:    1220778
    - Статус индексации:   completed


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    - Лексический поиск:   BM25S (C/NumPy, быстрый)


In [35]:
query = "Кто такие брони в констексте мультсериала my little pony?"
# top_parents — сколько цельных разделов статьи выдать для ответа
# score_threshold — порог отсечения (если статья не про это, вернёт False)
docs, is_relevant = retr1.search_and_rerank(
    query=query,
    top_parents=5,
    score_threshold=0.3,
    rerank=True
)
print(f"Найдена релевантная информация? {is_relevant}\n")
for i, doc in enumerate(docs, 1):
    print(f"[{i}] Статья: {doc.metadata.get('title')} | Раздел: {doc.metadata.get('section')}")
    print(f"Уверенность (Cross-Encoder Score): {doc.metadata.get('rerank_score')}")
    print("-" * 50)
    print(doc.page_content)

Найдена релевантная информация? True

[1] Статья: Старое обсуждение:My Little Pony | Раздел: Старое обсуждение:My Little Pony
Уверенность (Cross-Encoder Score): 0.8805
--------------------------------------------------
Вот еще что забыл сказать, по поводу возможного добавления раздела «Дальнейшая судьба MLP». Все, что от нас на данный момент требуется, это запастись попкорном и ждать либо полнометражку (тут я примерю на себя маску [[Капитан Очевидность|Капитана Очевидности]] и напомню, что нас должна интересовать полнометражка именно с ПОНИ, а не EG 4: Legend of Everfree), либо 7-ой сезон сериала (существование которого на уровне «слухи, порожденные двусмысленными намеками от актеров озвучки»).
: А как дела с разделом /mlp/ на Дваче Эйч-Кей?
::Не стоит беспокоится за цветных коней. У них уже сложилcя круг почитателей и собственная субкультура.Скорее всего, будет как с каким-нибудь стартреком.Тролодрочеры постепенно прекратят вайпать ими на имеджбордах, половина фонатов забудет, но свой

In [34]:


# Проверяем:
print(generate_with_hf(query,docs))

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


В контексте мультсериала My Little Pony, "Брони" - это фэндом, который активно следит за развитием персонажей и истории, создает собственные произведения, включая фанарт и фанфиксы. Они также активно участвуют в дискуссиях и обсуждениях на различных форумах и имиджбордах, связанных с мультсериалом. Брони относятся к числу поклонников мультсериала My Little Pony, и их деятельность включает в себя не только просмотр мультфильмов, но и создание дополнительного контента, который дополняет и расширяет мир мультсериала.
